# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [84]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a151a364ab', 'id': 'chatcmpl-DuuurWrI7pN4HfCvR8EnPPfTgkkCE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02bf-6233-70e1-be96-a39260a770e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [85]:
from langchain_core.tools import tool

In [86]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [87]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [88]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [89]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [90]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuusqgVSCBFtevsSamqLkvl5VZl2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02bf-6621-70b2-b8bd-a9d941bf931f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_vn1KX1rH1csQbcK0vGR9mtq3', 'type': 'tool_call'}

In [91]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_vn1KX1rH1csQbcK0vGR9mtq3',
  'type': 'tool_call'}]

In [92]:
tool_call = response.tool_calls[0]
tool_call["name"]

'get_current_time'

In [93]:
tool_dict[tool_call["name"]]

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x16cddcae0>)

In [94]:
tool_dict[tool_call["name"]].invoke(tool_call)

Asia/Seoul (서울) 현재시각 2026-06-26 16:05:26 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 16:05:26 ', name='get_current_time', tool_call_id='call_vn1KX1rH1csQbcK0vGR9mtq3')

In [95]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '서울'}
Asia/Seoul (서울) 현재시각 2026-06-26 16:05:26 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuusqgVSCBFtevsSamqLkvl5VZl2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02bf-6621-70b2-b8bd-a9d941bf931f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_vn1KX1rH1csQbcK0vGR9mtq3', 'type': 'tool_call'}

In [96]:
llm_with_tools.invoke(messages)

AIMessage(content='서울은 지금 2026년 6월 26일 16시 5분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 189, 'total_tokens': 211, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuutdUGZU8wZXVemU2ISK0Qczvgx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02bf-6aed-74a1-a84b-2f4575480cb6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 189, 'output_tokens': 22, 'total_tokens': 211, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [97]:
messages.append(llm_with_tools.invoke(messages))

In [98]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuusqgVSCBFtevsSamqLkvl5VZl2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02bf-6621-70b2-b8bd-a9d941bf931f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_vn1KX1rH1csQbcK0vGR9mtq3', 'type': 'tool_call'}

### 주가 예제

In [99]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [100]:
# 재실행 시 중복 메시지 방지: 서울 대화(5개)까지만 유지
messages = messages[:5]

messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [101]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuusqgVSCBFtevsSamqLkvl5VZl2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02bf-6621-70b2-b8bd-a9d941bf931f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_vn1KX1rH1csQbcK0vGR9mtq3', 'type': 'tool_call'}

In [102]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_Xr84hoQ88jiIri7A3AaIuNwk',
  'type': 'tool_call'}]

In [103]:
!pip install tabulate

In [104]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print("통과")
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

통과
{'ticker': 'TSLA', 'period': '1mo'}


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DuuusqgVSCBFtevsSamqLkvl5VZl2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02bf-6621-70b2-b8bd-a9d941bf931f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_vn1KX1rH1csQbcK0vGR9mtq3', 'type': 'tool_call'}

In [105]:
llm_with_tools.invoke(messages)

AIMessage(content='테슬라(TSLA)의 주가를 한 달 전과 비교해보면, 한 달 전인 2026년 5월 26일의 종가가 433.59달러였고, 현재(2026년 6월 25일)의 종가는 375.12달러입니다. 따라서 테슬라의 주가는 한 달 전에 비해 내렸습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 1667, 'total_tokens': 1752, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_bd1115524b', 'id': 'chatcmpl-DuuuyKKN7UVIbVf7kInBMyD3nCNcA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02bf-7f91-7ac0-b1ea-9a8f7d17b484-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1667, 'output_tokens': 85, 'total_tokens': 1752, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning'

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

In [1]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
]


In [4]:
from pdf_summary_langchain import summarize_pdf


tools = [summarize_pdf]
tool_dict = {"summarize_pdf":summarize_pdf}


In [5]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model='gpt-4o-mini')
llm_with_tools = llm.bind_tools(tools)


In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 109, 'total_tokens': 151, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_303afd94e3', 'id': 'chatcmpl-DuvLWOceyqJbORRJ1DYxXIsoOxUM4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02d8-9934-7693-97b5-009122dc03fa-0', tool_calls=[{'name': 'summarize_pdf', 'args': {'pdf_path': '/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/5일차/samples/videoanomaly.pdf'}, 'id': 'call_4aY0BbflvrI4kx02qezWMzLO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 42, 'total_tokens': 151, 'input_token_details': {'audio': 0,

In [7]:
response=llm_with_tools.invoke(messages)

In [8]:
messages.append(response)

In [9]:
for tool_call in response.tool_calls:
  out=tool_dict[tool_call['name']].invoke(tool_call)


입력받은 pdf_path = /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/5일차/samples/videoanomaly.pdf


In [10]:
out

ToolMessage(content='# INFP: INdustrial Video Anomaly Detection via Frequency Prioritization\n\n## 저자의 문제 인식 및 주장\n저자들은 산업 비디오에서 발생할 수 있는 이상 탐지를 위한 새로운 방법론을 제안하고 있다. 기존의 비디오 이상 탐지 방법들은 여러 정상 샘플 사이에서 이상을 식별하는 데 초점을 맞춰 전체 이미지를 분석하는 경향이 있다. 그러나 산업 환경에서는 제품이 단일 유형으로 움직이며 고정된 경로를 따라 이동하기 때문에, 세밀한 지역적 특징을 캡처할 필요가 있다. 저자들은 기존 방법이 산업 비디오에 직접 적용될 경우, 제품의 희소한 분포 및 조명 변화에 대한 취약성을 증가시킨다고 주장한다. 이를 해결하기 위해 INFP라는 인코더-디코더 프레임워크를 제안하는데, 이는 비디오의 주파수 영역 특징을 학습하여 모델의 강인성을 강화한다. 또한, 이동하는 객체와 정적 배경 간의 차이를 활용하는 경로 필터와 주파수 도메인 특징을 공간-시간 특성과 융합하는 다기능 융합 모듈을 통해 조명 변화의 영향을 줄이려 한다. 마지막으로, 저자들은 제안된 방법이 IPAD 데이터 세트에서 기존 방법들에 비해 우수한 성능을 보였음을 강조한다.\n\n## 저자 소개\nQianzi Yu, Kai Zhu, Yang Cao, 및 Yu Kang은 모두 중국 과학기술대학교와 헴페이 종합 국가과학센터의 인공지능 연구소에서 활동 중인 연구자들이다. 이들은 비디오 이상 탐지 및 인공지능 분야에서 다양한 연구 결과를 발표하며, 특히 산업 환경에서의 비디오 분석 문제를 해결하기 위한 신기술 개발에 주력하고 있다.', name='summarize_pdf', tool_call_id='call_1nyfkD7rLesJoqaEVF9fXfTZ')

In [11]:
messages.append(out)

In [12]:
llm_with_tools.invoke(messages)

AIMessage(content='문서의 저자는 Qianzi Yu, Kai Zhu, Yang Cao, 및 Yu Kang입니다. 이들은 모두 중국 과학기술대학교와 헴페이 종합 국가과학센터의 인공지능 연구소에서 활동 중인 연구자들입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 559, 'total_tokens': 616, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_303afd94e3', 'id': 'chatcmpl-DuvNM08PYZzBBCNyVUMYHnBkIa3T8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02da-5965-77d0-bba9-9826f096355e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 559, 'output_tokens': 57, 'total_tokens': 616, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})